# 第09课 - 元认知设计模式


## 设置

本笔记本演示了使用 Microsoft Agent Framework 的元认知（Metacognition）设计模式。

**先决条件：**
- 已通过环境变量配置 Azure OpenAI 部署
- Azure CLI 已完成身份验证 (`az login`)


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## 什么是元认知？

元认知是 **关于思维的思考**。在 AI 代理的语境中，这意味着构建能够：

- **自我反思** 审视它们自己的输出和推理过程
- **检测错误** 并优雅地恢复，而不是默默失败
- **评估** 它们的响应是否完整且有帮助
- **适应** 当初始方法不起作用时调整策略（例如，回退到备份系统）

一个具备元认知的代理不只是回答问题 — 它会监控自己的表现并即时调整。


## 主工具和备用工具

一种常见的元认知模式是 **回退策略**。代理会先尝试主工具；如果它失败（例如，404 错误），代理会识别该失败并透明地切换到备用工具。

这反映了现实世界的系统，其中主服务可能不可用，代理必须在选择替代路径之前自我诊断问题。

下面我们定义两个航班查询工具：
- **主工具** — 覆盖巴黎、东京和巴塞罗那
- **备用工具** — 覆盖柏林、悉尼和纽约市


In [ ]:
@tool(approval_mode="never_require")
def get_flight_times(
    destination: Annotated[str, "The destination city"]
) -> str:
    """Get available flight times for a destination (primary source)."""
    flights = {
        "Paris": "Departures: 08:00, 12:30, 17:45 — from $350",
        "Tokyo": "Departures: 11:00, 23:30 — from $890",
        "Barcelona": "Departures: 07:15, 14:00, 19:30 — from $280",
    }
    if destination in flights:
        return flights[destination]
    raise Exception(f"404: No flights found for {destination} in primary system")


@tool(approval_mode="never_require")
def get_flight_times_backup(
    destination: Annotated[str, "The destination city"]
) -> str:
    """Get available flight times from backup system (used when primary fails)."""
    backup_flights = {
        "Berlin": "Departures: 09:00, 16:00 — from $220",
        "Sydney": "Departures: 22:00 — from $1200",
        "New York City": "Departures: 06:00, 10:30, 15:00, 20:00 — from $450",
    }
    return backup_flights.get(
        destination,
        f"No flights found for {destination} in any system. Please try again later.",
    )

## 具有错误恢复的自我反思代理

下面的代理被指示首先尝试主飞行系统，识别故障，并透明地回退到备用系统。在每次响应之后，它会简要地自我评估是否完全回答了用户的问题。


In [ ]:
agent = await provider.create_agent(
    tools=[get_flight_times, get_flight_times_backup],
    name="FlightBookingAgent",
    instructions="""You are a flight booking agent with self-reflection capabilities.

When looking up flights:
1. Try the primary flight system first (get_flight_times)
2. If the primary system fails (404 error), acknowledge the error and try the backup system (get_flight_times_backup)
3. Always explain to the user what happened — be transparent about fallbacks
4. If both systems fail, apologize and suggest alternatives

After each response, briefly evaluate whether your answer was complete and helpful.""",
)

# Test with a destination in primary system
print("=== Test 1: Destination in primary system ===")
response = await agent.run(
    "What flights are available to Paris?",
    )
print(response)

# Test with a destination only in backup system
print("\n=== Test 2: Destination only in backup system ===")
response = await agent.run(
    "What flights are available to Berlin?",
    )
print(response)

## 自我评估模式

元认知的另一个方面是 **自我评估**: 一个独立的代理（或在第二遍中同一代理）会审查回复的完整性、准确性和有用性。

下面我们创建一个 `ResponseEvaluator` 代理，对旅行代理的回复在三个维度上进行评分。


In [ ]:
evaluation_agent = await provider.create_agent(
    tools=[get_flight_times, get_flight_times_backup],
    name="ResponseEvaluator",
    instructions="""You are a quality evaluator for travel agent responses.
Given a travel question and the agent's response, evaluate:
1. Completeness: Did it answer all parts of the question? (1-5)
2. Accuracy: Is the information correct? (1-5)
3. Helpfulness: Would a traveler find this useful? (1-5)
Provide a brief evaluation with scores and one suggestion for improvement.""",
)

# Evaluate the agent's response from Test 1
eval_prompt = f"""Question: What flights are available to Paris?
Agent Response: {response}

Please evaluate the above response."""

evaluation = await evaluation_agent.run(eval_prompt)
print("=== Self-Evaluation ===")
print(evaluation)

## 摘要

在本课中，您学习了如何使用 Microsoft 代理框架 构建 **元认知代理**：

- **自我反思**: 代理会监控自身的推理并透明地传达发生了什么。
- **带回退的错误恢复**: 一种主/备工具模式，代理检测到失败（例如 404 错误）时会自动尝试替代来源。
- **自我评估**: 一个独立的评估者代理，会对响应在完整性、准确性和有用性方面进行评分。

这些模式使代理更健壮、透明且值得信赖——这是生产部署的关键特性。


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
免责声明：
本文件已使用 AI 翻译服务 [Co-op Translator](https://github.com/Azure/co-op-translator) 进行翻译。尽管我们努力确保准确性，但请注意自动翻译可能包含错误或不准确之处。原始语言的原文应被视为权威来源。对于关键信息，建议由专业人工翻译。我们不对因使用本翻译而引起的任何误解或曲解承担责任。
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
